# Experiment 3 — AES-128
Round-wise transformation analysis: SubBytes → ShiftRows → MixColumns → AddRoundKey

## AES Constants (S-Box, Round Constants)

In [ ]:
SBOX=[0x63,0x7c,0x77,0x7b,0xf2,0x6b,0x6f,0xc5,0x30,0x01,0x67,0x2b,0xfe,0xd7,0xab,0x76,
      0xca,0x82,0xc9,0x7d,0xfa,0x59,0x47,0xf0,0xad,0xd4,0xa2,0xaf,0x9c,0xa4,0x72,0xc0,
      0xb7,0xfd,0x93,0x26,0x36,0x3f,0xf7,0xcc,0x34,0xa5,0xe5,0xf1,0x71,0xd8,0x31,0x15,
      0x04,0xc7,0x23,0xc3,0x18,0x96,0x05,0x9a,0x07,0x12,0x80,0xe2,0xeb,0x27,0xb2,0x75,
      0x09,0x83,0x2c,0x1a,0x1b,0x6e,0x5a,0xa0,0x52,0x3b,0xd6,0xb3,0x29,0xe3,0x2f,0x84,
      0x53,0xd1,0x00,0xed,0x20,0xfc,0xb1,0x5b,0x6a,0xcb,0xbe,0x39,0x4a,0x4c,0x58,0xcf,
      0xd0,0xef,0xaa,0xfb,0x43,0x4d,0x33,0x85,0x45,0xf9,0x02,0x7f,0x50,0x3c,0x9f,0xa8,
      0x51,0xa3,0x40,0x8f,0x92,0x9d,0x38,0xf5,0xbc,0xb6,0xda,0x21,0x10,0xff,0xf3,0xd2,
      0xcd,0x0c,0x13,0xec,0x5f,0x97,0x44,0x17,0xc4,0xa7,0x7e,0x3d,0x64,0x5d,0x19,0x73,
      0x60,0x81,0x4f,0xdc,0x22,0x2a,0x90,0x88,0x46,0xee,0xb8,0x14,0xde,0x5e,0x0b,0xdb,
      0xe0,0x32,0x3a,0x0a,0x49,0x06,0x24,0x5c,0xc2,0xd3,0xac,0x62,0x91,0x95,0xe4,0x79,
      0xe7,0xc8,0x37,0x6d,0x8d,0xd5,0x4e,0xa9,0x6c,0x56,0xf4,0xea,0x65,0x7a,0xae,0x08,
      0xba,0x78,0x25,0x2e,0x1c,0xa6,0xb4,0xc6,0xe8,0xdd,0x74,0x1f,0x4b,0xbd,0x8b,0x8a,
      0x70,0x3e,0xb5,0x66,0x48,0x03,0xf6,0x0e,0x61,0x35,0x57,0xb9,0x86,0xc1,0x1d,0x9e,
      0xe1,0xf8,0x98,0x11,0x69,0xd9,0x8e,0x94,0x9b,0x1e,0x87,0xe9,0xce,0x55,0x28,0xdf,
      0x8c,0xa1,0x89,0x0d,0xbf,0xe6,0x42,0x68,0x41,0x99,0x2d,0x0f,0xb0,0x54,0xbb,0x16]
INV_SBOX=[0]*256
for i,v in enumerate(SBOX): INV_SBOX[v]=i
RCON=[0x01,0x02,0x04,0x08,0x10,0x20,0x40,0x80,0x1b,0x36]
print("AES constants loaded ✓")


## GF(2⁸) Arithmetic & State Operations

In [ ]:
def gf_mul(a,b):
    p=0
    for _ in range(8):
        if b&1: p^=a
        hi=a&0x80; a=(a<<1)&0xff
        if hi: a^=0x1b
        b>>=1
    return p

def b2state(b): return [[b[r+4*c] for c in range(4)] for r in range(4)]
def state2b(s): return bytes(s[r][c] for c in range(4) for r in range(4))

def sub_bytes(s):   return [[SBOX[s[r][c]] for c in range(4)] for r in range(4)]
def isub_bytes(s):  return [[INV_SBOX[s[r][c]] for c in range(4)] for r in range(4)]
def shift_rows(s):  return [[(s[r][(c+r)%4]) for c in range(4)] for r in range(4)]
def ishift_rows(s): return [[(s[r][(c-r)%4]) for c in range(4)] for r in range(4)]

def mix_cols(s):
    o=[[0]*4 for _ in range(4)]
    for c in range(4):
        col=[s[r][c] for r in range(4)]
        o[0][c]=gf_mul(2,col[0])^gf_mul(3,col[1])^col[2]^col[3]
        o[1][c]=col[0]^gf_mul(2,col[1])^gf_mul(3,col[2])^col[3]
        o[2][c]=col[0]^col[1]^gf_mul(2,col[2])^gf_mul(3,col[3])
        o[3][c]=gf_mul(3,col[0])^col[1]^col[2]^gf_mul(2,col[3])
    return o

def imix_cols(s):
    o=[[0]*4 for _ in range(4)]
    for c in range(4):
        col=[s[r][c] for r in range(4)]
        o[0][c]=gf_mul(0xe,col[0])^gf_mul(0xb,col[1])^gf_mul(0xd,col[2])^gf_mul(0x9,col[3])
        o[1][c]=gf_mul(0x9,col[0])^gf_mul(0xe,col[1])^gf_mul(0xb,col[2])^gf_mul(0xd,col[3])
        o[2][c]=gf_mul(0xd,col[0])^gf_mul(0x9,col[1])^gf_mul(0xe,col[2])^gf_mul(0xb,col[3])
        o[3][c]=gf_mul(0xb,col[0])^gf_mul(0xd,col[1])^gf_mul(0x9,col[2])^gf_mul(0xe,col[3])
    return o

def ark(s,rk): return [[s[r][c]^rk[r][c] for c in range(4)] for r in range(4)]

print("AES state operations defined ✓")


## Key Expansion

In [ ]:
def key_expansion(key):
    w=list(key)
    for i in range(4,44):
        t=w[(i-1)*4:i*4]
        if i%4==0:
            t=[SBOX[t[1]],SBOX[t[2]],SBOX[t[3]],SBOX[t[0]]]; t[0]^=RCON[i//4-1]
        w+=[w[(i-4)*4+j]^t[j] for j in range(4)]
    return [b2state(bytes(w[r*16:(r+1)*16])) for r in range(11)]

print("Key expansion defined ✓")


## AES Encrypt / Decrypt Block

In [ ]:
def aes_enc(block, key, verbose=False):
    rk=key_expansion(key); s=ark(b2state(block),rk[0])
    if verbose: print(f"  Initial ARK : {state2b(s).hex().upper()}")
    for rnd in range(1,11):
        s=sub_bytes(s)
        if verbose: print(f"  Rnd {rnd:2d} SB  : {state2b(s).hex().upper()}")
        s=shift_rows(s)
        if verbose: print(f"  Rnd {rnd:2d} SR  : {state2b(s).hex().upper()}")
        if rnd<10:
            s=mix_cols(s)
            if verbose: print(f"  Rnd {rnd:2d} MC  : {state2b(s).hex().upper()}")
        s=ark(s,rk[rnd])
        if verbose: print(f"  Rnd {rnd:2d} ARK : {state2b(s).hex().upper()}")
    return state2b(s)

def aes_dec(block, key):
    rk=key_expansion(key); s=ark(b2state(block),rk[10])
    for rnd in range(9,-1,-1):
        s=ishift_rows(s); s=isub_bytes(s); s=ark(s,rk[rnd])
        if rnd>0: s=imix_cols(s)
    return state2b(s)

def pkcs7_pad(d,bs=16): p=bs-(len(d)%bs); return d+bytes([p]*p)
def pkcs7_unpad(d): return d[:-d[-1]]
def aes_encrypt(pt,key): return b''.join(aes_enc(pkcs7_pad(pt)[i:i+16],key) for i in range(0,len(pkcs7_pad(pt)),16))
def aes_decrypt(ct,key): return pkcs7_unpad(b''.join(aes_dec(ct[i:i+16],key) for i in range(0,len(ct),16)))

print("AES encrypt/decrypt defined ✓")


## NIST FIPS-197 Appendix C.1 Verification + Round Trace

In [ ]:
key   = bytes.fromhex('000102030405060708090a0b0c0d0e0f')
block = bytes.fromhex('00112233445566778899aabbccddeeff')
expected = bytes.fromhex('69c4e0d86a7b0430d8cdb78070b4c55a')   # NIST FIPS-197

print(f"Key       : {key.hex().upper()}")
print(f"Plaintext : {block.hex().upper()}")
print()
ct = aes_enc(block, key, verbose=True)
print()
print(f"Ciphertext : {ct.hex().upper()}")
print(f"Expected   : {expected.hex().upper()}")
assert ct == expected
print("✓ NIST FIPS-197 C.1 vector verified")


## Roundtrip Test

In [ ]:
key2 = b'SecretKey1234567'
pt2  = b'AES Experiment 3'
ct2  = aes_encrypt(pt2, key2)
dt2  = aes_decrypt(ct2, key2)
print(f"Plaintext  : {pt2}")
print(f"Encrypted  : {ct2.hex().upper()}")
print(f"Decrypted  : {dt2}")
assert dt2 == pt2
print("✓ Roundtrip verified")
